# Naive RAG Pipeline for Collins Aerospace Documents

This notebook implements:
- Document loading
- Chunking
- Embeddings + FAISS
- Retrieval QA
- Evaluation metrics

**Compatible with Google Colab**

In [ ]:
!pip install langchain langchain-community langchain-openai faiss-cpu tiktoken scikit-learn

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your_api_key_here"

## Load Documents

In [ ]:
from langchain.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "./collins_docs",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()
print(f"Loaded {len(documents)} pages")

## Chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)
print(f"Total chunks: {len(docs)}")

## Embeddings + Vector Store

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

## Query Example

In [ ]:
query = "What are the key avionics systems described in Collins Aerospace manuals?"
answer = qa_chain.run(query)
print("Answer:\n", answer)

## Evaluation Metrics

In [ ]:
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

def recall_at_k(retrieved_docs, ground_truth_docs):
    retrieved_ids = set([doc.metadata.get("source", "") for doc in retrieved_docs])
    gt_ids = set(ground_truth_docs)
    return len(retrieved_ids & gt_ids) / len(gt_ids) if gt_ids else 0

def precision_at_k(retrieved_docs, ground_truth_docs):
    retrieved_ids = set([doc.metadata.get("source", "") for doc in retrieved_docs])
    gt_ids = set(ground_truth_docs)
    return len(retrieved_ids & gt_ids) / len(retrieved_ids) if retrieved_ids else 0

def f1_score(pred, gt):
    pred_tokens = pred.lower().split()
    gt_tokens = gt.lower().split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * (precision * recall) / (precision + recall)

def semantic_similarity(a, b, embeddings):
    vec_a = embeddings.embed_query(a)
    vec_b = embeddings.embed_query(b)
    return cosine_similarity([vec_a], [vec_b])[0][0]

## Evaluation Dataset

In [ ]:
eval_data = [
    {
        "question": "What communication systems are used?",
        "answer": "Collins Aerospace uses integrated communication systems including VHF and satellite communication.",
        "source_docs": ["doc1.pdf"]
    }
]

## Evaluation Loop

In [ ]:
results = []
for item in eval_data:
    pred = qa_chain.run(item["question"])
    retrieved_docs = retriever.get_relevant_documents(item["question"])
    rec = recall_at_k(retrieved_docs, item["source_docs"])
    prec = precision_at_k(retrieved_docs, item["source_docs"])
    f1 = f1_score(pred, item["answer"])
    results.append({
        "question": item["question"],
        "recall": rec,
        "precision": prec,
        "f1": f1
    })

avg_recall = sum(r["recall"] for r in results) / len(results)
avg_precision = sum(r["precision"] for r in results) / len(results)
avg_f1 = sum(r["f1"] for r in results) / len(results)

print("Average Recall:", avg_recall)
print("Average Precision:", avg_precision)
print("Average F1:", avg_f1)